In [1]:
import cv2
import time
import math
from collections import deque
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

try:
    import serial
except ImportError:
    serial = None

try:
    from pynput.keyboard import Controller, Key
    keyboard = Controller()
    MEDIA_KEYS = {
        "PLAY_PAUSE": Key.media_play_pause,
        "NEXT_TRACK": Key.media_next,
        "PREV_TRACK": Key.media_previous,
    }
except ImportError:
    keyboard = None
    MEDIA_KEYS = {}

# ---------------- settings ----------------
CAMERA_INDEX = 0
SERIAL_PORT = "COM5"          # change to your ESP32 port
BAUD_RATE = 115200

SMOOTH_FRAMES = 4
TRAIL_LENGTH = 40
MIN_STEP_PX = 2

CIRCLE_WINDOW = 0.6
CIRCLE_MIN_TURN = 2.5         # radians turned inside the window
CIRCLE_CONSISTENCY = 0.75
CIRCLE_MIN_SPEED = 150        # px per second
MOVING_WINDOW = 0.15
STOP_DELAY = 0.25
ROT_RESEND = 0.3
BRIGHT_STEP_RAD = 0.5
HEADING_WINDOW = 0.1

SWIPE_WINDOW = 0.4
SWIPE_MIN_DIST = 0.3          # fraction of frame width
SWIPE_STRAIGHT = 2.0
SWIPE_MAX_TURN = 1.0
SWIPE_COOLDOWN = 0.8

HOLD_TIME = 0.5
STILL_SPEED = 120
SIGN_COOLDOWN = 1.0

SIGN_ACTIONS = {
    "Open_Palm": "PWR",
    "Fist": "PLAY_PAUSE",
    "Thumbs_Up": "NEXT_TRACK",
    "Thumbs_Down": "PREV_TRACK",
}
# ------------------------------------------

last_cmd = ""
ser = None


def dist(a, b):
    return math.hypot(a.x - b.x, a.y - b.y)


def get_sign(hand):
    thumb = dist(hand[4], hand[17]) > dist(hand[3], hand[17])
    index = dist(hand[8], hand[0]) > dist(hand[6], hand[0])
    middle = dist(hand[12], hand[0]) > dist(hand[10], hand[0])
    ring = dist(hand[16], hand[0]) > dist(hand[14], hand[0])
    pinky = dist(hand[20], hand[0]) > dist(hand[18], hand[0])
    fingers = [thumb, index, middle, ring, pinky]

    if index and middle and not ring and not pinky:
        return "Beam"
    if index and not middle and not ring and not pinky:
        return "Point"
    if fingers == [True, True, True, True, True]:
        return "Open_Palm"
    if fingers == [False, False, False, False, False]:
        return "Fist"
    if fingers == [True, False, False, False, False]:
        return "Thumbs_Up" if hand[4].y < hand[2].y else "Thumbs_Down"
    return "None"


def recent(trail, now, window):
    return [p for p in trail if now - p[2] <= window]


def analyze(points):
    vecs = []
    for i in range(1, len(points)):
        dx = points[i][0] - points[i - 1][0]
        dy = points[i][1] - points[i - 1][1]
        if math.hypot(dx, dy) >= MIN_STEP_PX:
            vecs.append((dx, dy))

    turn = 0.0
    pos = neg = 0
    for i in range(1, len(vecs)):
        x1, y1 = vecs[i - 1]
        x2, y2 = vecs[i]
        a = math.atan2(x1 * y2 - y1 * x2, x1 * x2 + y1 * y2)
        turn += a
        if a > 0.02:
            pos += 1
        elif a < -0.02:
            neg += 1

    n = pos + neg
    consistency = max(pos, neg) / n if n else 0.0
    path = sum(math.hypot(dx, dy) for dx, dy in vecs)
    dt = points[-1][2] - points[0][2] if len(points) > 1 else 0.0
    speed = path / dt if dt > 0 else 0.0
    return turn, consistency, speed


def heading(trail, now):
    pts = recent(trail, now, HEADING_WINDOW)
    if len(pts) < 2:
        return None
    dx = pts[-1][0] - pts[0][0]
    dy = pts[-1][1] - pts[0][1]
    if math.hypot(dx, dy) < MIN_STEP_PX * 2:
        return None
    return math.atan2(dy, dx)


def wrap(a):
    while a > math.pi:
        a -= 2 * math.pi
    while a < -math.pi:
        a += 2 * math.pi
    return a


def send(cmd):
    global last_cmd
    last_cmd = cmd
    print("->", cmd)
    if ser is not None:
        try:
            ser.write((cmd + "\n").encode())
        except Exception as e:
            print("Serial error:", e)


def run_action(action):
    global last_cmd
    if action == "PWR":
        send("PWR")
        return
    last_cmd = action
    print("->", action)
    if keyboard is not None and action in MEDIA_KEYS:
        keyboard.tap(MEDIA_KEYS[action])


def open_serial():
    if serial is None:
        print("pyserial not installed, running without ESP32")
        return None
    try:
        s = serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=0)
        time.sleep(2)
        print("Connected to ESP32 on", SERIAL_PORT)
        return s
    except Exception as e:
        print("No ESP32 on", SERIAL_PORT, "-", e)
        return None


ser = open_serial()

options = vision.GestureRecognizerOptions(
    base_options=python.BaseOptions(model_asset_path="gesture_recognizer.task"),
    running_mode=vision.RunningMode.VIDEO,
    num_hands=1,
)
recognizer = vision.GestureRecognizer.create_from_options(options)

cap = cv2.VideoCapture(CAMERA_INDEX)
start_time = time.time()

raw = deque(maxlen=SMOOTH_FRAMES)
trail = deque(maxlen=TRAIL_LENGTH)

motion_mode = None
rot_dir = None
last_circle_time = 0.0
last_rot_send = 0.0
bri_acc = 0.0
prev_heading = None
last_swipe = 0.0
held_sign = "None"
held_since = 0.0
held_fired = False
last_action_time = 0.0

while True:
    ok, frame = cap.read()
    if not ok:
        break

    frame = cv2.flip(frame, 1)
    height, width, _ = frame.shape
    now = time.time()

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    timestamp = int((now - start_time) * 1000)
    result = recognizer.recognize_for_video(mp_image, timestamp)

    sign = "None"

    if result.hand_landmarks:
        hand = result.hand_landmarks[0]

        for number, point in enumerate(hand):
            x = int(point.x * width)
            y = int(point.y * height)
            cv2.circle(frame, (x, y), 5, (0, 255, 0), -1)
            cv2.putText(frame, str(number), (x + 5, y - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)

        tip_x = int(hand[8].x * width)
        tip_y = int(hand[8].y * height)
        raw.append((tip_x, tip_y))
        sx = sum(p[0] for p in raw) / len(raw)
        sy = sum(p[1] for p in raw) / len(raw)
        trail.append((sx, sy, now))
        cv2.circle(frame, (tip_x, tip_y), 12, (0, 0, 255), -1)

        thumb_x = int(hand[4].x * width)
        thumb_y = int(hand[4].y * height)
        cv2.circle(frame, (thumb_x, thumb_y), 12, (255, 0, 0), -1)

        sign = get_sign(hand)

        turn, consistency, speed = analyze(recent(trail, now, CIRCLE_WINDOW))
        _, _, fast_speed = analyze(recent(trail, now, MOVING_WINDOW))
        circling = (abs(turn) >= CIRCLE_MIN_TURN
                    and consistency >= CIRCLE_CONSISTENCY
                    and speed >= CIRCLE_MIN_SPEED
                    and fast_speed >= CIRCLE_MIN_SPEED)
        clockwise = turn > 0   # y points down, so positive = clockwise on screen

        # ---- circles: 1 finger = rotate, 2 fingers = brightness ----
        if motion_mode is None:
            if circling and sign in ("Point", "Beam"):
                motion_mode = "ROTATE" if sign == "Point" else "BRIGHT"
                last_circle_time = now
                bri_acc = 0.0
                prev_heading = None
                if motion_mode == "ROTATE":
                    rot_dir = "R" if clockwise else "L"
                    send("ROT:" + rot_dir)
                    last_rot_send = now
        else:
            if circling:
                last_circle_time = now

            if motion_mode == "ROTATE":
                if circling:
                    new_dir = "R" if clockwise else "L"
                    if new_dir != rot_dir:
                        rot_dir = new_dir
                        send("ROT:" + rot_dir)
                        last_rot_send = now
                if now - last_circle_time > STOP_DELAY:
                    send("ROT:S")
                    motion_mode = None
                    rot_dir = None
                elif now - last_rot_send >= ROT_RESEND:
                    send("ROT:" + rot_dir)
                    last_rot_send = now
            else:
                h = heading(trail, now)
                if h is not None:
                    if prev_heading is not None:
                        bri_acc += wrap(h - prev_heading)
                    prev_heading = h
                while bri_acc >= BRIGHT_STEP_RAD:
                    send("BRI:+")
                    bri_acc -= BRIGHT_STEP_RAD
                while bri_acc <= -BRIGHT_STEP_RAD:
                    send("BRI:-")
                    bri_acc += BRIGHT_STEP_RAD
                if now - last_circle_time > STOP_DELAY:
                    motion_mode = None

        # ---- swipe: 1 finger, fast and straight sideways ----
        if motion_mode is None and sign == "Point" and now - last_swipe >= SWIPE_COOLDOWN:
            sp = recent(trail, now, SWIPE_WINDOW)
            if len(sp) >= 4:
                dx = sp[-1][0] - sp[0][0]
                dy = sp[-1][1] - sp[0][1]
                s_turn, _, _ = analyze(sp)
                if (abs(dx) >= SWIPE_MIN_DIST * width
                        and abs(dx) >= SWIPE_STRAIGHT * abs(dy)
                        and abs(s_turn) <= SWIPE_MAX_TURN):
                    send("COL:+" if dx > 0 else "COL:-")
                    last_swipe = now
                    trail.clear()
                    raw.clear()

        # ---- static signs: hold still for HOLD_TIME ----
        if sign != held_sign:
            held_sign = sign
            held_since = now
            held_fired = False
        if motion_mode is not None or fast_speed > STILL_SPEED:
            held_since = now

        if sign in SIGN_ACTIONS and not held_fired:
            progress = min(1.0, (now - held_since) / HOLD_TIME)
            cv2.rectangle(frame, (10, 130), (10 + int(200 * progress), 145), (0, 255, 255), -1)
            cv2.rectangle(frame, (10, 130), (210, 145), (255, 255, 255), 1)
            if progress >= 1.0 and now - last_action_time >= SIGN_COOLDOWN:
                run_action(SIGN_ACTIONS[sign])
                held_fired = True
                last_action_time = now
    else:
        if motion_mode == "ROTATE":
            send("ROT:S")
        motion_mode = None
        rot_dir = None
        trail.clear()
        raw.clear()
        held_sign = "None"
        held_fired = False

    pts = list(trail)
    for i in range(1, len(pts)):
        thick = 1 + int(6 * i / len(pts))
        cv2.line(frame, (int(pts[i - 1][0]), int(pts[i - 1][1])),
                 (int(pts[i][0]), int(pts[i][1])), (0, 255, 255), thick)

    cv2.putText(frame, "Sign: " + sign, (10, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
    cv2.putText(frame, "Mode: " + (motion_mode or "-"), (10, 75),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    cv2.putText(frame, "Sent: " + last_cmd, (10, 110),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)
    cv2.putText(frame, "ESP32: " + ("connected" if ser else "offline"), (10, height - 15),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)

    cv2.imshow("Gesture Lamp", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

if motion_mode == "ROTATE":
    send("ROT:S")
if ser is not None:
    ser.close()
cap.release()
cv2.destroyAllWindows()

pyserial not installed, running without ESP32
-> PWR
-> ROT:L
-> ROT:L
-> ROT:S
-> ROT:R
-> ROT:R
-> ROT:S
-> ROT:R
-> ROT:R
-> ROT:S
-> ROT:R
-> ROT:R
-> ROT:R
-> ROT:S
-> PWR
-> NEXT_TRACK
-> PLAY_PAUSE
-> NEXT_TRACK


In [1]:
%pip install pynput

Using cached pynput-1.8.2-py2.py3-none-any.whl (92 kB)
Note: you may need to restart the kernel to use updated packages.
